In [4]:
import os, sys, sqlite3

print("Python: ", sys.version)
print("SQLite library: ", sqlite3.sqlite_version)

DB_PATH = "experiment.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
print("DB file will be at: ", os.path.abspath(DB_PATH))

def table(conn):
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()
    return [r[0] for r in rows]

Python:  3.13.16 (main, Oct 19 2026, 14:03:22) [Clang 21.0.0 (clang-2100.3.34.2)]
SQLite library:  3.54.0
DB file will be at:  /Users/zherao/projects/frontend/experiments/05-database-integrity-lab/backend/experiments/experiment.db


In [5]:
conn = sqlite3.connect(DB_PATH)
print("isolation_level: ", repr(conn.isolation_level))
print("autocommit: ", getattr(conn, "autocommit", "n/a (Python < 3.12)"))

print("in_transaction before CREATE: ", conn.in_transaction)
conn.execute("CREATE TABLE clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
print("in_transaction after CREATE: ", conn.in_transaction)

conn.commit()
conn.close()
print("File exists after close: ", os.path.exists(DB_PATH), " | size: ", os.path.getsize(DB_PATH), " bytes")

isolation_level:  ''
autocommit:  -1
in_transaction before CREATE:  False
in_transaction after CREATE:  False
File exists after close:  True  | size:  8192  bytes


In [6]:
conn = sqlite3.connect(DB_PATH, autocommit=False)
print("isolation_level: ", repr(conn.isolation_level))
print("autocommit: ", getattr(conn, "autocommit", "n/a (Python < 3.12)"))

print("in_transaction before CREATE: ", conn.in_transaction)
conn.execute("CREATE TABLE IF NOT EXISTS clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
print("in_transaction after CREATE: ", conn.in_transaction)

conn.commit()
conn.close()
print("File exists after close: ", os.path.exists(DB_PATH), " | size: ", os.path.getsize(DB_PATH), " bytes")

isolation_level:  ''
autocommit:  False
in_transaction before CREATE:  True
in_transaction after CREATE:  True
File exists after close:  True  | size:  8192  bytes


In [7]:
conn = sqlite3.connect(DB_PATH)
print("Tables after reconnect: ", table(conn))
conn.close()

Tables after reconnect:  ['clients']


In [8]:
os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, autocommit=False)
conn.execute("CREATE TABLE clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
print("in_transaction after CREATE:", conn.in_transaction)
conn.close()

conn = sqlite3.connect(DB_PATH, autocommit=False)
print("Tables after reopen (no commit): ", table(conn))
conn.close()

in_transaction after CREATE: True
Tables after reopen (no commit):  []


In [13]:
os.remove(DB_PATH)

f1 = sqlite3.connect(DB_PATH, autocommit=False)
f2 = sqlite3.connect(DB_PATH, autocommit=False)

f1.execute("CREATE TABLE IF NOT EXISTS clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")

print("f1 sees: ", table(f1))
print("f2 sees: ", table(f2))


f1.close()
f2.close()

f1 sees:  ['clients']
f2 sees:  []


In [14]:
os.remove(DB_PATH)

f1 = sqlite3.connect(DB_PATH, autocommit=False)
f2 = sqlite3.connect(DB_PATH, autocommit=False)

f1.execute("CREATE TABLE IF NOT EXISTS clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
f1.commit()

print("f1 sees: ", table(f1))
print("f2 sees: ", table(f2))


f1.close()
f2.close()

f1 sees:  ['clients']
f2 sees:  ['clients']


In [15]:
m1 = sqlite3.connect(":memory:")
m1.execute("CREATE TABLE clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
print("m1 sees: ", table(m1))

m2 = sqlite3.connect(":memory:")
print("m2 sees: ", table(m2))

m1.close()
m1 = sqlite3.connect(":memory:")
print("m1 after close + reconnect sees: ", table(m1))

m1.close()
m2.close()

m1 sees:  ['clients']
m2 sees:  []
m1 after close + reconnect sees:  []
